# 5장 — 지도 그리기: 차원 축소 🗜️

PCA를 **numpy만으로** 재발견하고, 축소된 지도가 어떻게 거짓말하는지 재현하고,
**진짜 t-SNE**까지 직접 돌려 봅니다.

**이 노트북의 목표**
1. 축을 돌려 투영 분산이 최대인 방향 찾기 → 공분산 고유벡터(진짜 PCA)와 일치 확인
2. 8차원 toy 임베딩을 PCA로 2D 투영 — 클러스터가 살아남는지 확인
3. 지도의 거짓말: 투영이 거리 순위를 뒤집는 사례를 3D→2D로 재현
4. **진짜 t-SNE 실행**(scikit-learn) — 같은 데이터가 설정(perplexity)에 따라 다른 지도가 되는 것을 목격

> UMAP 실행은 별도 패키지(umap-learn)가 필요해 다루지 않지만, 읽는 법은 t-SNE와 같습니다.

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch5')

## 1. 실험 ① — PCA 재발견: 그림자가 가장 넓게 지는 축

투영(그림자) 좌표 = 점 · 축 (내적). 축을 1°씩 돌리며 그림자의 분산을 잽니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(2026)

# 주축 30° 방향으로 길게 퍼진 2D 점 구름
deg = np.radians(30)
u_true = np.array([np.cos(deg), np.sin(deg)])
w_true = np.array([-np.sin(deg), np.cos(deg)])
pts = np.outer(rng.standard_normal(60) * 2.0, u_true) + np.outer(rng.standard_normal(60) * 0.55, w_true)

# 축을 돌려 가며 그림자 분산 재기
angles = np.arange(180)
variances = [np.var(pts @ np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])) for a in angles]
best = int(np.argmax(variances))
print(f'그림자 분산이 최대인 축: {best}°  (설계값 30°)')
assert abs(best - 30) <= 5, '분산 최대 축은 설계한 주축(30°) 근처여야'

# 진짜 PCA — 공분산 행렬의 고유벡터 (1부 8장)
cov = np.cov(pts.T)
eigvals, eigvecs = np.linalg.eigh(cov)
pc1 = eigvecs[:, -1]                       # 가장 큰 고유값의 고유벡터 = 제1주성분
pca_deg = np.degrees(np.arctan2(pc1[1], pc1[0])) % 180
print(f'공분산 고유벡터(PCA)의 방향: {pca_deg:.1f}°')
assert abs(pca_deg - best) <= 3, '축 돌리기의 답 = PCA의 답'
print('\n✔ 확인: "분산 최대 방향 찾기" = PCA — 1부 8장의 주성분이 이것')

## 2. 실험 ② — 8차원 임베딩을 2D 지도로

8차원 toy 임베딩(3개 클러스터)을 PCA 상위 2축에 투영해 지도를 그립니다.

In [ ]:
# 8차원에 클러스터 3개 (중심을 서로 떨어뜨려 배치)
centers = np.array([
    [4, 0, 0, 0, 2, 0, 0, 0],   # 인사 문서
    [0, 4, 0, 2, 0, 0, 0, 0],   # 경비 문서
    [0, 0, 4, 0, 0, 2, 0, 0],   # 보안 문서
], dtype=float)
X = np.vstack([c + 0.4 * rng.standard_normal((12, 8)) for c in centers])
labels = np.repeat(['인사', '경비', '보안'], 12)

# PCA 상위 2축 투영 (numpy만으로)
Xc = X - X.mean(axis=0)
_, vecs = np.linalg.eigh(np.cov(Xc.T))
map2d = Xc @ vecs[:, -2:]                  # 분산 1·2위 축

plt.figure(figsize=(5.5, 4))
for name, color in zip(['인사', '경비', '보안'], ['tab:blue', 'tab:orange', 'tab:green']):
    m = labels == name
    plt.scatter(map2d[m, 0], map2d[m, 1], c=color, label=name)
plt.legend(); plt.title('8D -> 2D PCA map'); plt.grid(alpha=0.3); plt.show()

# 2D 지도에서도 무리 안 < 무리 밖인지 정량 확인
def avg_pair_dist(A, B):
    return float(np.mean([np.linalg.norm(a - b) for a in A for b in B]))
inner = np.mean([avg_pair_dist(map2d[labels == g], map2d[labels == g]) for g in ['인사', '경비', '보안']])
outer = avg_pair_dist(map2d[labels == '인사'], map2d[labels == '경비'])
assert outer > inner * 2, '잘 만든 지도에선 클러스터가 살아남아야'
print(f'무리 안 평균 {inner:.2f} vs 무리 밖 평균 {outer:.2f} — 클러스터 생존 ✔')

## 3. 실험 ③ — 지도의 거짓말: 투영이 순위를 뒤집는다

그림자는 한 방향(여기선 z)을 통째로 버립니다 — 버린 방향에 있던 거리도 함께 사라지죠.
3D → 2D 로 가장 단순하게 재현합니다.

In [ ]:
P = {  # 3차원 진실
    'A': np.array([0.0, 0.0, 0.0]),
    'B': np.array([4.0, 0.0, 0.5]),
    'C': np.array([2.0, 3.0, 6.0]),   # z(버려질 방향)로 멀리 떨어진 점
}

def d3(a, b): return float(np.linalg.norm(P[a] - P[b]))
def d2(a, b): return float(np.linalg.norm(P[a][:2] - P[b][:2]))   # z를 버린 그림자 거리

print(' 쌍    실제(3D)   지도(2D, z 버림)')
for pair in [('A', 'B'), ('A', 'C'), ('B', 'C')]:
    print(f'  {pair[0]}-{pair[1]}    {d3(*pair):5.2f}      {d2(*pair):5.2f}')

# 실제로는 A-B(4.03)가 A-C(7.00)보다 가깝지만, 지도에선 A-C(3.61)가 A-B(4.00)보다 가깝다!
assert d3('A', 'B') < d3('A', 'C'), '3D 진실: A-B가 더 가깝다'
assert d2('A', 'C') < d2('A', 'B'), '2D 지도: A-C가 더 가까워 보인다 — 순위 역전!'
print('\n⚠️ 확인: 버린 차원(z)에 있던 거리가 사라지며 순위가 뒤집혔다')
print('   → 이웃은 믿되 지도 위 거리로 결론 내리지 말 것, 검색은 원 공간에서(6장)')

## 4. 실험 ④ — 진짜 t-SNE: 같은 데이터, 다른 지도

실험 ②의 8차원 데이터를 이번엔 **이웃 보존형** 지도(t-SNE)로 그립니다 — 설정값
perplexity("이웃을 몇 명쯤으로 볼 것인가")만 바꿔서 두 장을.

본문의 "읽는 법"이 그대로 검증됩니다: **무리 짓기(이웃)는 두 지도 모두 지켜지지만,
무리 사이의 배치·거리 비율은 지도마다 달라집니다.**

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
maps = {}
for ax, perp in zip(axes, [3, 15]):
    Y = TSNE(n_components=2, perplexity=perp, random_state=0,
             init='pca', learning_rate='auto').fit_transform(X)
    maps[perp] = Y
    for name, color in zip(['인사', '경비', '보안'], ['tab:blue', 'tab:orange', 'tab:green']):
        m = labels == name
        ax.scatter(Y[m, 0], Y[m, 1], c=color, label=name)
    ax.set_title(f't-SNE (perplexity={perp})'); ax.grid(alpha=0.3)
axes[0].legend(); plt.tight_layout(); plt.show()

# ① 이웃(무리 짓기)은 지켜지는가 — 각 점의 최근접 3점이 같은 클러스터인 비율
def neighbor_purity(Y):
    hits = 0
    for i in range(len(Y)):
        nn = np.argsort(np.linalg.norm(Y - Y[i], axis=1))[1:4]
        hits += np.mean(labels[nn] == labels[i])
    return hits / len(Y)

for perp, Y in maps.items():
    purity = neighbor_purity(Y)
    print(f'perplexity={perp:2d}: 이웃 보존(3-NN 순도) {purity:.0%}')
    assert purity > 0.8, '이웃(클러스터)은 어느 설정에서도 지켜져야 — 이웃 보존형의 약속'

# ② 무리 "사이"는? — 클러스터 중심 간 거리의 비율이 지도마다 달라진다
def centroid_gaps(Y):
    cs = {g: Y[labels == g].mean(axis=0) for g in ['인사', '경비', '보안']}
    d = lambda a, b: float(np.linalg.norm(cs[a] - cs[b]))
    base = d('인사', '경비')
    return {'인사-경비': 1.0, '인사-보안': d('인사', '보안') / base, '경비-보안': d('경비', '보안') / base}

for perp, Y in maps.items():
    g = centroid_gaps(Y)
    print(f'perplexity={perp:2d}: 클러스터 간 거리 비율 (인사-경비=1 기준) '
          f"인사-보안 {g['인사-보안']:.2f} / 경비-보안 {g['경비-보안']:.2f}")

print('\n✔ 확인: 무리 짓기는 두 지도 모두 유지 — 그러나 무리 사이 배치·비율은 설정 따라 달라진다')
print('   → 이웃은 믿되, 클러스터 간 거리로 결론 내리지 말 것 (지도 여러 장을 보는 게 정석)')

## 5. 관찰 정리 + 🤖 AI 연결

- **PCA** = 그림자(투영 = 내적)의 분산이 최대인 축 고르기 — 1부 8장의 주성분, 선형·결정적.
- 잘 만든 2D 지도에서도 살아남는 것은 **클러스터(이웃 관계)** — 관찰·디버깅에 유용.
- 그러나 **버린 차원의 거리들은 사라진다** — 지도 위 거리·크기·모양으로 결론 내리지 말 것.
- **진짜 t-SNE로 확인한 것**(실험 ④): 이웃 보존(3-NN 순도)은 설정과 무관하게 유지되지만,
  클러스터 사이 배치·거리 비율은 perplexity 하나로 달라진다 — 같은 데이터, 다른 지도.
  UMAP도 읽는 법은 같다(설정값 이름만 n_neighbors).
- 검색은 항상 **원 공간(수백 차원)** 에서 — 지도는 눈을 위한 요약일 뿐.

**다음 챕터**: 6장 — 이웃 찾기. 수만 개 문서에서 1등을 순식간에 찾는 법:
kNN 전수조사의 비용 → 근사 최근접 이웃(ANN) → **벡터 DB의 정체**.